# 36 CPU audio impact baseline

既存の `raw_videos` / `clips_v1` を再利用し、打球音の contact-centered impact window から raw / enhanced の音声特徴、predictions、metrics、研究用 HTML/CSV/SVG report を作ります。

Creates: `features/{audio_*_feature_id}/`, `datasets/audio_feature_samples/{audio_*_run_id}/`, `predictions/{audio_*_run_id}/`, `reports/audio_impact/{audio_*_run_id}/`。

Required inputs: `manifests/bbe_events_v1.parquet`, `clips/{full_run_id}/clips_v1.parquet`, clip files with audio tracks。If `features/{audio_audit_feature_id}/audio_valid_clips_v1.parquet` exists, this notebook uses it to avoid training on empty audio windows.

Next: `37_gpu_audio_separation_and_embeddings.ipynb` for Demucs/HF audio embeddings, then `38_cpu_audio_fusion_compare.ipynb`.

In [ ]:
from pathlib import Path
import json
import os
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or already mounted:', exc)

os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff_with_audio')

repo_dir = Path(os.environ['BATTING_CODE_ROOT'])
if not (repo_dir / 'src' / 'sport_pipeline').exists():
    repo_dir = Path.cwd()
sys.path.insert(0, str(repo_dir / 'src'))

profile_path = repo_dir / 'configs' / 'runs' / os.environ['BASEBALL_VISION_RUN_PROFILE']
profile = json.loads(profile_path.read_text(encoding='utf-8'))
BASE_DIR = Path(profile['paths']['base_dir'])
CACHE_DIR = Path(profile['paths'].get('cache_dir', '/content/cache/baseball_vision'))
RUN_IDS = profile['run_ids']
NS = profile['artifact_namespace']
AUDIO_EXEC = profile['execution']['audio_impact']
print('repo_dir=', repo_dir)
print('BASE_DIR=', BASE_DIR)
print('full_run_id=', RUN_IDS['full_run_id'])

In [ ]:
required = [
    BASE_DIR / 'manifests' / 'bbe_events_v1.parquet',
    BASE_DIR / 'clips' / RUN_IDS['full_run_id'] / 'clips_v1.parquet',
]
for path in required:
    print(('OK      ' if path.exists() else 'MISSING '), path)
if not all(path.exists() for path in required):
    raise FileNotFoundError('Required artifact missing. Run Statcast/download/CV preprocessing notebooks first.')

In [ ]:
from sport_pipeline.audio.impact import run_audio_impact_baseline

MAX_CLIPS = AUDIO_EXEC.get('max_clips')
OUTPUT_FORMAT = 'parquet'
COMMON = dict(
    base_dir=BASE_DIR,
    clip_run_id=RUN_IDS['full_run_id'],
    max_clips=MAX_CLIPS,
    require_non_empty=AUDIO_EXEC.get('require_non_empty', True),
    output_suffix='.' + OUTPUT_FORMAT,
    cache_dir=CACHE_DIR,
    cache_inputs=AUDIO_EXEC.get('cache_inputs', True),
    cache_num_workers=AUDIO_EXEC.get('cache_num_workers', 4),
    cache_min_free_disk_gb=AUDIO_EXEC.get('cache_min_free_disk_gb', 30),
    cache_max_file_mb=AUDIO_EXEC.get('cache_max_file_mb', 256),
    valid_audio_only=AUDIO_EXEC.get('valid_audio_only', True),
)
audit_valid = BASE_DIR / 'features' / NS.get('audio_audit_feature_id', RUN_IDS.get('audio_presence_audit_id', 'audio_presence_mlb_2024_2026_v2')) / 'audio_valid_clips_v1.parquet'
if AUDIO_EXEC.get('use_audio_audit_valid_clips', True) and audit_valid.exists():
    COMMON['clips_path'] = audit_valid
    print('Using audio-valid clip manifest:', audit_valid)
else:
    print('Audio-valid clip manifest not found; filtering empty windows during extraction:', audit_valid)

outputs = {}
if AUDIO_EXEC.get('run_raw_default', True):
    outputs['raw'] = run_audio_impact_baseline(
        prediction_run_id=RUN_IDS['audio_raw_run_id'],
        audio_feature_id=NS['audio_raw_feature_id'],
        preprocessing_mode='raw',
        **COMMON,
    )
if AUDIO_EXEC.get('run_enhanced_default', True):
    outputs['enhanced'] = run_audio_impact_baseline(
        prediction_run_id=RUN_IDS['audio_enhanced_run_id'],
        audio_feature_id=NS['audio_enhanced_feature_id'],
        preprocessing_mode='enhanced',
        **COMMON,
    )
print(json.dumps({k: {kk: str(vv) for kk, vv in v.items()} for k, v in outputs.items()}, indent=2, ensure_ascii=False))

In [ ]:
for branch, branch_outputs in outputs.items():
    print('\n==', branch, '==')
    for key in ['predictions', 'metrics', 'audio_report_html', 'audio_system_diagram', 'audio_correlation_chart']:
        value = branch_outputs.get(key)
        if value:
            print(f'{key}: {value}')
print('\nNext notebook: notebooks/37_gpu_audio_separation_and_embeddings.ipynb')